## 100% Full Data PMT Observables and Event Rates for the ANNIE NCQE Analysis

In [93]:
print('Loading Packages...\n')
%matplotlib osx
import sys, os          
import uproot
import awkward as ak
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm import trange
import numpy as np
import awkward as ak
from scipy.optimize import curve_fit
import pandas as pd
from collections import defaultdict
import scipy.stats
import matplotlib.gridspec as gridspec
from scipy.stats import chi2_contingency
import random
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
import matplotlib.patches as mpatches
font = {'family' : 'serif', 'size' : 14 }
mpl.rc('font', **font)
mpl.rcParams['mathtext.fontset'] = 'cm' # Set the math font to Computer Modern
mpl.rcParams['legend.fontsize'] = 16
# Set default figure and axes face colors to white
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# in the '42_44/' directory, there are 4169 files
# 5.518e16 POT per world file, therefore:
WORLD_POT = 5.518e16
N_files = 4169
POT = N_files*WORLD_POT    # MC POT
print('MC POT:', POT)

# tank dimensions:
radius = 1.524
half_height = 1.98

FV_radius = 0.7
FV_half_height = 1

bunch_time_cutoff = 1560
time_to_prompt_end = 258       # MC approximate time from end of spill to 2us cutoff (chain last MC bunch to observed data last bunch)

centers = np.array([30.55620074944292, 50.23596632813184, 68.03027546306257, 87.65149609251257, 106.38480577249247,
                    125.28116343989417, 144.32309666797107, 163.1397586239191, 181.97778394194054, 201.06274310271073,
                    220.16343853423476, 238.30288809166953, 258.38514881200354, 276.8146398611303, 296.0884356855441,
                    314.35016323142816, 333.72419652049575, 352.9437944077438, 371.260196759933, 390.08407553949974,
                    409.44913114396746, 427.94715055258985, 447.770917943728, 465.9339836950206, 484.7186644716273,
                    504.4287698808072, 523.0814143742606, 542.3141201678939, 561.2254420066273, 579.8111323601904,
                    597.9945900965694, 618.0700927506651, 636.5621477504429, 655.7874418315001, 674.4764151744993,
                    693.6905256979796, 712.4819567023824, 731.0362230741698, 750.2744038347329, 769.2102243656747,
                    788.2453849590231, 807.0011262101622, 825.9067358626738, 845.3047059731147, 864.0653260882543,
                    883.0323070032683, 901.6429155761502, 920.6808374077715, 939.6712240835648, 958.6990235603214,
                    977.72856819438, 996.2787338241341, 1015.08703056768, 1034.3375283354696, 1053.9759092673044,
                    1072.052984198524, 1091.129322973113, 1110.3389344568238, 1128.965140612168, 1148.3985550156349,
                    1166.982403872106, 1186.220604492118, 1204.4016122615021, 1223.770899436999, 1242.691800785879,
                    1261.9422162459668, 1280.3202276072477, 1299.702530449622, 1318.214516959161, 1337.6134384701843,
                    1356.5308849686603, 1375.0603202255497, 1393.91335999822, 1413.0147004160135, 1431.9961602521892,
                    1451.5391995383018, 1470.035773429933, 1488.7824718348304, 1507.6311809765648, 1526.7701548484445,
                    1545.2663160965799])

# spill duration for data
spill_start_22 = 206
spill_end_22 = 1739

spill_start_23 = 189
spill_end_23 = 1719

centers_data_23 = np.array([198.3700379, 215.5394396, 234.2243011, 251.5523355, 272.0954178,
                            290.6241403, 308.5785171, 328.6072512, 347.7788392, 366.7606742,
                            385.8724421, 403.4237083, 422.6212047, 441.7513664, 460.8075196,
                            479.1954063, 499.1118531, 518.0524434, 536.2365549, 555.4753724,
                            574.4827687, 592.706705, 612.5075049, 631.5683442, 650.1400917,
                            669.2300559, 688.445265, 706.8814584, 725.5179244, 745.1280734,
                            763.1263233, 782.1646795, 801.9846119, 820.4792799, 839.8275515,
                            858.4967685, 876.8179439, 896.6128644, 915.5773794, 933.9912385,
                            953.7637369, 971.8604713, 992.8093626, 1009.494428, 1028.17604,
                            1047.085112, 1067.172794, 1084.712851, 1104.77286, 1123.612654,
                            1143.01596, 1161.339718, 1180.141252, 1200.09213, 1218.150494,
                            1237.658053, 1256.417415, 1274.927169, 1293.037241, 1312.298147,
                            1331.977857, 1351.217822, 1370.450652, 1388.800942, 1407.751471,
                            1426.765537, 1445.82194, 1464.015855, 1483.30137, 1502.409606,
                            1520.862787, 1539.567404, 1560.027696, 1577.842804, 1597.580137,
                            1616.224347, 1635.150248, 1654.172443, 1672.743926, 1691.044861, 1710.421766])

centers_data_22 = np.array([215.7494731, 234.5402877, 252.6960547, 271.922027, 290.5568422,
                            309.4619782, 328.3685976, 347.1585617, 366.1694916, 384.6981959,
                            403.679617, 422.6118675, 441.7254035, 460.6980149, 479.8100942,
                            498.6283802, 517.4495414, 536.5487368, 555.6267941, 574.5979937,
                            593.0192376, 612.3368162, 630.8397525, 649.8131052, 669.2991329,
                            687.9304207, 706.7121484, 725.7727685, 744.8524787, 763.6331082,
                            782.3583396, 801.7439406, 820.2399223, 839.2074824, 858.2099129,
                            876.8574359, 896.0077987, 915.1144969, 934.2965022, 953.2212701,
                            971.751155, 990.5106562, 1009.956462, 1028.540103, 1047.63785,
                            1066.887371, 1085.900211, 1104.166114, 1123.524265, 1142.494186,
                            1161.263316, 1180.440215, 1199.068621, 1217.817441, 1236.616553,
                            1256.531327, 1274.887169, 1293.907079, 1313.039446, 1331.26644,
                            1350.901016, 1369.621886, 1388.363869, 1407.764603, 1425.900837,
                            1445.194635, 1464.29313, 1483.001877, 1502.552265, 1520.924771,
                            1539.732641, 1558.053482, 1577.955086, 1596.717582, 1616.106719,
                            1634.94341, 1653.694115, 1672.695675, 1691.851104, 1710.382291, 1729.90487])

bunch_sigma = 3.59

def reco_energy(pe):
    return (pe + 2.90)/5.90

def bunch_cutting(cluster_time, centers):
    avg_diff = 0.5081
    return np.any(np.abs( (centers - avg_diff) - cluster_time) < bunch_sigma)

def pairwise_relative_direction(hitX, hitY, hitZ, hitPE, hitT):
    
    hitX = np.array(hitX)
    hitY = np.array(hitY)
    hitZ = np.array(hitZ)
    hitPE = np.array(hitPE)
    hitT = np.array(hitT)

    # Combine and sort by time
    hits = list(zip(hitX, hitY, hitZ, hitPE, hitT))
    hits.sort(key=lambda h: h[4])  # sort by time
    
    #filtered_hits = hits[:4]
    
    mini = min(hitT)
    filtered_hits = []
    # Process remaining hits
    for hit in hits:
        x_i, y_i, z_i, pe_i, t_i = hit

        if (t_i - mini) > 13:
            continue
            
        filtered_hits.append(hit)

    # Extract filtered values
    filtX = np.array([h[0] for h in filtered_hits])
    filtY = np.array([h[1] for h in filtered_hits])
    filtZ = np.array([h[2] for h in filtered_hits])
    filtPE = np.array([h[3] for h in filtered_hits])
    
    n = len(filtX)
    if n < 2:
        return np.array([0.0, 0.0, 0.0])
    
    vec = np.zeros(3)
    for i in range(n):
        for j in range(i+1, n):
            dx = filtX[j] - filtX[i]
            dy = filtY[j] - filtY[i]
            dz = filtZ[j] - filtZ[i]
            r = np.sqrt(dx**2 + dy**2 + dz**2)
            if r == 0:
                continue
            
            # Direction between hits
            dvec = np.array([dx, dy, dz]) / r
            weight = filtPE[i] * filtPE[j]
            vec += weight * dvec
    
    norm = np.linalg.norm(vec)
    if norm == 0:
        return np.array([0.0, 0.0, 0.0])
    return vec / norm

def is_inside_rotated_parabola(Z_val, Y_val, a=-0.45, c=0.4, b=0.0, theta_deg=275):
    # Convert degrees to radians
    theta = np.deg2rad(theta_deg)
    
    # Inverse rotation (world → local)
    u =  Z_val*np.cos(theta) + Y_val*np.sin(theta)
    v = -Z_val*np.sin(theta) + Y_val*np.cos(theta)
    
    # Equation in local coords: v = a*u^2 + b*u + c
    # If opening is along +v, "inside" means v >= parabola
    return v < a*u**2 + b*u + c


df = pd.read_csv('../../MC_GENIE_Jul_2025/FullTankPMTGeometry.csv')
channel_number = []; PMT_type = []
for i in range(len(df['channel_num'])):
    channel_number.append(df['channel_num'][i])
    PMT_type.append(df['PMT_type'][i])
    

print('done')

Loading Packages...

MC POT: 2.3004542e+20
done


In [84]:
# Mini-cell w/ pre-loading to determine the total number of true NCQE signal events in the FV

total_weighted = 0.0
total_raw      = 0          # unweighted count, useful sanity check

# MC directory used for this analysis can be found here: /pnfs/annie/persistent/users/doran/GENIE_reweight_MC_ToolChain/42_44/
# These files were not passed through STV, but do contain weights
MC_directory = '42_44/'
file_names = [f for f in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, f))]

for file_idx in trange(len(file_names), desc='Loading MC files'):
    path = os.path.join(MC_directory, file_names[file_idx])
    root = uproot.open(path)

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']
    vx  = S['trueNuIntxVtx_X'].array(library='np')
    vy  = S['trueNuIntxVtx_Y'].array(library='np')
    vz  = S['trueNuIntxVtx_Z'].array(library='np')
    nc  = S['trueNC'].array(library='np')
    qel = S['trueQEL'].array(library='np')
    nu  = S['trueFSLPdg'].array(library='np')
    tz  = S['trueTargetZ'].array(library='np')
    wt  = S['weight_TunedCentralValue_UBGenie'].array(library='np')
    
    vx_cm = vx / 100
    vy_cm = vy / 100 + 0.1446
    vz_cm = vz / 100 - 1.681
    r2    = vx_cm**2 + vz_cm**2
    fv_flags  = (r2 < FV_radius**2) & (np.abs(vy_cm) < FV_half_height)
    
    # ── single combined selection mask ────────────────────────────────────────
    mask = (
        fv_flags          &
        (nc  == 1)        &
        (qel == 1)        &
        (tz  == 8)        &
        ((nu == 14) | (nu == 12))
    )

    wt_cv = np.concatenate(wt).astype(float)

    total_weighted += wt_cv[mask].sum()
    total_raw      += mask.sum()
    

print(f'\nRaw signal events  : {total_raw}')
print(f'Weighted signal sum: {total_weighted:.4f}')
print('\ndone')

Loading MC files: 100%|█████████████████████| 4169/4169 [06:55<00:00, 10.04it/s]


Raw signal events  : 10100
Weighted signal sum: 10100.0000

done


In [3]:
# Load MC samples

MC_directory = '../42_44/'
file_names = [f for f in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, f))]

# scalar accumulators
MC_cluster_number    = []
MC_cluster_time      = []
MC_cluster_charge    = []
MC_cluster_Qb        = []
MC_cluster_Hits      = []
MC_TankMRDCoinc      = []
MC_NoVeto            = []
MC_recoVtx           = []
MC_recoVty           = []
MC_recoVtz           = []
MC_Vtx               = []
MC_Vty               = []
MC_Vtz               = []
MC_Vtt               = []
MC_bunchTimes        = []
MC_External          = []
MC_FV                = []
MC_NC                = []
MC_QEL               = []
MC_MEC               = []
MC_PDG               = []
MC_Z                 = []
MC_file_index        = []   # which file each cluster came from
MC_event_number      = []
MC_CV_weight         = []

# jagged accumulators
MC_hitX  = []
MC_hitY  = []
MC_hitZ  = []
MC_hitT  = []
MC_hitPE = []
MC_hitID = []
MC_MRD_hitT  = []
MC_FMV_hitT  = []

for file_idx in trange(len(file_names), desc='Loading MC files'):
    path = os.path.join(MC_directory, file_names[file_idx])
    root = uproot.open(path)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    MC_cluster_number.append(T['clusterNumber'].array(library='np'))
    MC_cluster_charge.append(T['clusterPE'].array(library='np'))
    MC_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    MC_cluster_Hits.append(T['clusterHits'].array(library='np'))
    MC_cluster_time.append(T['clusterTime'].array(library='np'))
    MC_bunchTimes.append(T['bunchTimes'].array(library='np'))
    MC_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    MC_NoVeto.append(T['NoVeto'].array(library='np'))
    MC_recoVtx.append(T['recoLeastSqVtxX'].array(library='np'))
    MC_recoVty.append(T['recoLeastSqVtxY'].array(library='np'))
    MC_recoVtz.append(T['recoLeastSqVtxZ'].array(library='np'))
    MC_event_number.append(CEN)
    MC_file_index.append(np.full(len(CEN), file_idx, dtype=np.int32))

    MC_hitX.append(T['hitX'].array())
    MC_hitY.append(T['hitY'].array())
    MC_hitZ.append(T['hitZ'].array())
    MC_hitT.append(T['hitT'].array())
    MC_hitPE.append(T['hitPE'].array())
    MC_hitID.append(T['hitDetID'].array())

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']
    vx  = S['trueNuIntxVtx_X'].array(library='np')
    vy  = S['trueNuIntxVtx_Y'].array(library='np')
    vz  = S['trueNuIntxVtx_Z'].array(library='np')

    vx_cm = vx / 100
    vy_cm = vy / 100 + 0.1446
    vz_cm = vz / 100 - 1.681
    r2    = vx_cm**2 + vz_cm**2
    ext_flags = (r2 > radius**2)    | (np.abs(vy_cm) > half_height)
    fv_flags  = (r2 < FV_radius**2) & (np.abs(vy_cm) < FV_half_height)

    # map trigger-level quantities onto clusters via CEN
    MC_Vtx.append(vx_cm[CEN])
    MC_Vty.append(vy_cm[CEN])
    MC_Vtz.append(vz_cm[CEN])
    MC_Vtt.append(S['trueNuIntxVtx_T'].array(library='np')[CEN])
    MC_External.append(ext_flags[CEN])
    MC_FV.append(fv_flags[CEN])
    MC_NC.append(S['trueNC'].array(library='np')[CEN])
    MC_QEL.append(S['trueQEL'].array(library='np')[CEN])
    MC_MEC.append(S['trueMEC'].array(library='np')[CEN])
    MC_PDG.append(S['trueFSLPdg'].array(library='np')[CEN])
    MC_Z.append(S['trueTargetZ'].array(library='np')[CEN])
    
    MC_CV_weight.append(S['weight_TunedCentralValue_UBGenie'].array(library='np')[CEN])

    mrd_t   = S['MRDhitT'].array()
    veto_t  = S['FMVhitT'].array()
    MC_MRD_hitT.append(mrd_t[CEN])
    MC_FMV_hitT.append(veto_t[CEN])

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

MC_cluster_number  = np.concatenate(MC_cluster_number)
MC_cluster_time    = np.concatenate(MC_cluster_time)
MC_cluster_charge  = np.concatenate(MC_cluster_charge)
MC_cluster_Qb      = np.concatenate(MC_cluster_Qb)
MC_cluster_Hits    = np.concatenate(MC_cluster_Hits)
MC_bunchTimes      = np.concatenate(MC_bunchTimes)
MC_TankMRDCoinc    = np.concatenate(MC_TankMRDCoinc)
MC_NoVeto          = np.concatenate(MC_NoVeto)
MC_recoVtx         = np.concatenate(MC_recoVtx)
MC_recoVty         = np.concatenate(MC_recoVty)
MC_recoVtz         = np.concatenate(MC_recoVtz)
MC_event_number    = np.concatenate(MC_event_number)
MC_file_index      = np.concatenate(MC_file_index)
MC_Vtx             = np.concatenate(MC_Vtx)
MC_Vty             = np.concatenate(MC_Vty)
MC_Vtz             = np.concatenate(MC_Vtz)
MC_Vtt             = np.concatenate(MC_Vtt)
MC_External        = np.concatenate(MC_External)
MC_FV              = np.concatenate(MC_FV)
MC_NC              = np.concatenate(MC_NC)
MC_QEL             = np.concatenate(MC_QEL)
MC_MEC             = np.concatenate(MC_MEC)
MC_PDG             = np.concatenate(MC_PDG)
MC_Z               = np.concatenate(MC_Z)
MC_CV_weight       = np.concatenate(MC_CV_weight)

MC_hitX     = ak.concatenate(MC_hitX)
MC_hitY     = ak.concatenate(MC_hitY)
MC_hitZ     = ak.concatenate(MC_hitZ)
MC_hitT     = ak.concatenate(MC_hitT)
MC_hitPE    = ak.concatenate(MC_hitPE)
MC_hitID    = ak.concatenate(MC_hitID)
MC_MRD_hitT = ak.concatenate(MC_MRD_hitT)
MC_FMV_hitT = ak.concatenate(MC_FMV_hitT)

print(f'POT: {POT:.3e}')
print(f'Total clusters: {len(MC_cluster_number)}')
print('\ndone')

Loading MC files: 100%|█████████████████████| 4169/4169 [09:02<00:00,  7.68it/s]



Managing arrays...

POT: 2.300e+20
Total clusters: 1064828

done


In [4]:
# on-beam data       

def extract_run_number(file_name):
    return int(file_name.split('_')[0][1:])  # Split by '_' and remove 'R' to get the run number

directory = '../FY22_23/'
file_names = [file for file in os.listdir(directory) if os.path.isfile(os.path.join(directory, file))]
print('There are: ', len(file_names), ' files\n')

# Extract all run numbers from the current files
run_numbers = [extract_run_number(f) for f in file_names]

# POT file
csv_file = '../POT_files/POT_trigger_FY22_23_summary.csv'
full_pot_df = pd.read_csv(csv_file)

# Filter the dataframe to only include runs that are in the <STRING>/ directory
pot_df = full_pot_df[full_pot_df['RUN'].isin(run_numbers)]

# extract total POT and triggers for only the matched runs
total_POT_TOR860 = sum(pot_df["TOR860_POTe12"])/(1e8)   # convert to e20 POT
total_triggers = sum(pot_df["TOTAL_TRIGGERS"])

cluster_time = []
cluster_charge = []
cluster_Hits = []
cluster_Qb = []
TankMRDCoinc = []
NoVeto = []
only_prompt_cluster = []
MRD_activity = []
Veto_activity = []
Vtx_reco = []
Vty_reco = []
Vtz_reco = []
hitX = []
hitY = []
hitZ = []
hitT = []
hitPE = []
runs = []

counter = 0
print('(', (counter), '/', len(file_names), ')')

for file_name in file_names:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 50 == 0:
        print('(', (counter), '/', len(file_names_ext), ')')
    
    with uproot.open(os.path.join(directory, file_name)) as file:
        
        Event = file["data"]
        
        runs.append(Event["run_number"].array(library="np"))
        
        # needed for both
        cluster_time.append(Event["cluster_time_BRF"].array(library="np"))
        cluster_charge.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto.append(Event["NoVeto"].array(library="np"))
        only_prompt_cluster.append(Event["only_prompt_cluster"].array(library="np"))
        MRD_activity.append(Event["MRD_activity"].array(library="np"))
        Veto_activity.append(Event["FMV_activity"].array(library="np"))
        Vtx_reco.append(Event["recoVtx"].array(library="np"))
        Vty_reco.append(Event["recoVty"].array(library="np"))
        Vtz_reco.append(Event["recoVtz"].array(library="np"))
        
        hitX.append(Event["hitX"].array())
        hitY.append(Event["hitY"].array())
        hitZ.append(Event["hitZ"].array())
        hitT.append(Event["hitT"].array())
        hitPE.append(Event["hitPE"].array())

        #print(len(cluster_time[-1]), "clusters")
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs = np.concatenate(runs)
cluster_time = np.concatenate(cluster_time)
cluster_charge = np.concatenate(cluster_charge)
cluster_Qb = np.concatenate(cluster_Qb)
cluster_Hits = np.concatenate(cluster_Hits)
TankMRDCoinc = np.concatenate(TankMRDCoinc)
NoVeto = np.concatenate(NoVeto)
only_prompt_cluster = np.concatenate(only_prompt_cluster)
MRD_activity = np.concatenate(MRD_activity)
Veto_activity = np.concatenate(Veto_activity)
Vtx_reco = np.concatenate(Vtx_reco)
Vty_reco = np.concatenate(Vty_reco)
Vtz_reco = np.concatenate(Vtz_reco)

# Jagged arrays
hitX = ak.concatenate(hitX)
hitY = ak.concatenate(hitY)
hitZ = ak.concatenate(hitZ)
hitT = ak.concatenate(hitT)
hitPE = ak.concatenate(hitPE)

print(total_triggers, 'triggers')
print(total_POT_TOR860, 'e20 POT\n')

There are:  389  files

( 0 / 389 )
( 1 / 389 )
( 2 / 389 )
( 3 / 389 )
( 4 / 389 )
( 5 / 389 )
( 6 / 389 )
( 7 / 389 )
( 8 / 389 )
( 9 / 389 )
( 10 / 389 )
( 11 / 389 )
( 12 / 389 )
( 13 / 389 )
( 14 / 389 )
( 15 / 389 )
( 16 / 389 )
( 17 / 389 )
( 18 / 389 )
( 19 / 389 )
( 20 / 389 )
( 21 / 389 )
( 22 / 389 )
( 23 / 389 )
( 24 / 389 )
( 25 / 389 )
( 26 / 389 )
( 27 / 389 )
( 28 / 389 )
( 29 / 389 )
( 30 / 389 )
( 31 / 389 )
( 32 / 389 )
( 33 / 389 )
( 34 / 389 )
( 35 / 389 )
( 36 / 389 )
( 37 / 389 )
( 38 / 389 )
( 39 / 389 )
( 40 / 389 )
( 41 / 389 )
( 42 / 389 )
( 43 / 389 )
( 44 / 389 )
( 45 / 389 )
( 46 / 389 )
( 47 / 389 )
( 48 / 389 )
( 49 / 389 )
( 50 / 389 )
( 51 / 389 )
( 52 / 389 )
( 53 / 389 )
( 54 / 389 )
( 55 / 389 )
( 56 / 389 )
( 57 / 389 )
( 58 / 389 )
( 59 / 389 )
( 60 / 389 )
( 61 / 389 )
( 62 / 389 )
( 63 / 389 )
( 64 / 389 )
( 65 / 389 )
( 66 / 389 )
( 67 / 389 )
( 68 / 389 )
( 69 / 389 )
( 70 / 389 )
( 71 / 389 )
( 72 / 389 )
( 73 / 389 )
( 74 / 389 )
( 75 / 389 

In [5]:
# off beam data

directory_ext = '../off_beam/'
file_names_ext = [file for file in os.listdir(directory_ext) if os.path.isfile(os.path.join(directory_ext, file))]
print('There are: ', len(file_names_ext), ' files\n')

# POT file
csv_file_ext = '../POT_files/offbeam_summary.csv'
pot_df_ext = pd.read_csv(csv_file_ext)

# extract total POT and triggers
total_ext_triggers = sum(pot_df_ext["TOTAL_TRIGGERS"])

cluster_time_ext = []
cluster_charge_ext = []
cluster_Hits_ext = []
cluster_Qb_ext = []
TankMRDCoinc_ext = []
NoVeto_ext = []
only_prompt_cluster_ext = []
MRD_activity_ext = []
Veto_activity_ext = []
Vtx_reco_ext = []
Vty_reco_ext = []
Vtz_reco_ext = []
hitX_ext = []
hitY_ext = []
hitZ_ext = []
hitT_ext = []
hitPE_ext = []
runs_ext = []

counter = 0
print('(', (counter), '/', len(file_names_ext), ')')

for file_name in file_names_ext:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 50 == 0:
        print('(', (counter), '/', len(file_names_ext), ')')
    
    with uproot.open(os.path.join(directory_ext, file_name)) as file:
        
        Event = file["data"]
        
        runs_ext.append(Event["run_number"].array(library="np"))
        
        # needed for both
        cluster_time_ext.append(Event["cluster_time"].array(library="np"))
        cluster_charge_ext.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb_ext.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits_ext.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc_ext.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto_ext.append(Event["NoVeto"].array(library="np"))
        only_prompt_cluster_ext.append(Event["only_prompt_cluster"].array(library="np"))
        MRD_activity_ext.append(Event["MRD_activity"].array(library="np"))
        Veto_activity_ext.append(Event["FMV_activity"].array(library="np"))
        Vtx_reco_ext.append(Event["recoVtx"].array(library="np"))
        Vty_reco_ext.append(Event["recoVty"].array(library="np"))
        Vtz_reco_ext.append(Event["recoVtz"].array(library="np"))
        
        hitX_ext.append(Event["hitX"].array())
        hitY_ext.append(Event["hitY"].array())
        hitZ_ext.append(Event["hitZ"].array())
        hitT_ext.append(Event["hitT"].array())
        hitPE_ext.append(Event["hitPE"].array())
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs_ext = np.concatenate(runs_ext)
cluster_time_ext = np.concatenate(cluster_time_ext)
cluster_charge_ext = np.concatenate(cluster_charge_ext)
cluster_Qb_ext = np.concatenate(cluster_Qb_ext)
cluster_Hits_ext = np.concatenate(cluster_Hits_ext)
TankMRDCoinc_ext = np.concatenate(TankMRDCoinc_ext)
NoVeto_ext = np.concatenate(NoVeto_ext)
only_prompt_cluster_ext = np.concatenate(only_prompt_cluster_ext)
MRD_activity_ext = np.concatenate(MRD_activity_ext)
Veto_activity_ext = np.concatenate(Veto_activity_ext)
Vtx_reco_ext = np.concatenate(Vtx_reco_ext)
Vty_reco_ext = np.concatenate(Vty_reco_ext)
Vtz_reco_ext = np.concatenate(Vtz_reco_ext)

# Jagged arrays
hitX_ext = ak.concatenate(hitX_ext)
hitY_ext = ak.concatenate(hitY_ext)
hitZ_ext = ak.concatenate(hitZ_ext)
hitT_ext = ak.concatenate(hitT_ext)
hitPE_ext = ak.concatenate(hitPE_ext)

print(total_ext_triggers, 'triggers')

There are:  389  files

( 0 / 389 )
( 25 / 389 )
( 50 / 389 )
( 75 / 389 )
( 100 / 389 )
( 125 / 389 )
( 150 / 389 )
( 175 / 389 )
( 200 / 389 )
( 225 / 389 )
( 250 / 389 )
( 275 / 389 )
( 300 / 389 )
( 325 / 389 )
( 350 / 389 )
( 375 / 389 )

Managing arrays...

24737351 triggers


In [162]:
# If doing the 1/3 scaling stats boost trick, extract scaling factor (see cell below for details)
print(min(MC_bunchTimes))
print(centers[25], centers[80])
print('Scaling =', (centers[80]-min(MC_bunchTimes))/(centers[25]-min(MC_bunchTimes)))

19.787590488471373
504.4287698808072 1545.2663160965799
Scaling = 3.1476457025810727


In [206]:
# make MC cuts

# We can be clever with our MC prediction
# Given we don't see any "delayed" clusters --> i.e. clusters > 200ns from the initial neutrino interaction,
# We can "boost" our MC statistics by NOT cutting on the first third of the spill
# Given the neutrino interactions are independent within the spill, we can scale
# the MC prediction with this trick as if we were cutting on the first ~1/3 of the spill
# In practice this involves cutting on bunch [80] rather than [25], then scaling the MC POT via:
# min(bunchTime) = 20ns, [25] = 504ns, [80] = 1545 --> scaling: (1545 - 20) / (504-20) = ~3.15

MC_reco_E = [[], [], [], [], [], []]
MC_CV_x = [[], [], [], [], [], []]
MC_CV_y = [[], [], [], [], [], []]
MC_CV_z = [[], [], [], [], [], []]
MC_bunch_times = [[], [], [], [], [], []]
MC_vertex_x = [[], [], [], [], [], []]
MC_vertex_y = [[], [], [], [], [], []]
MC_vertex_z = [[], [], [], [], [], []]
MC_hits = [[], [], [], [], [], []]

MC_weights_by_cat = [[], [], [], [], [], []]


# arrays to store events that are "outside" the 1-sigma bunch cuts
MC_outside_bunch_by_cat = [[], [], [], [], [], []]
MC_outside_weights_by_cat = [[], [], [], [], [], []]


NCQE_signal_events = 0    # total number of true signal events


for i in trange(len(MC_cluster_time), desc = 'applying NCQE selection cuts to MC...'):
    
    # ------ Has cluster ------- #
    # primary cluster
    if MC_cluster_number[i] != 0:
        continue
    # cluster is within the prompt window
    if MC_bunchTimes[i] > bunch_time_cutoff + time_to_prompt_end:
        continue
    # minimum 10 hits
    if MC_cluster_Hits[i] < 10:
        continue
    # ------ Has cluster ------- #
    
    # No FMV or MRD activity
    if MC_TankMRDCoinc[i] == 1 or MC_NoVeto[i] == 0 or len(MC_MRD_hitT[i]) != 0 or len(MC_FMV_hitT[i]) != 0:
        continue
        
    # Cluster time within 1.6 us beam spill
    if MC_bunchTimes[i] > bunch_time_cutoff:
        continue
        
    # Add earlier time cut
    # Since there are ~0 delayed events (outside of 200ns from the initial particle creation),
    # we can boost the MC statistics by "folding" the full spill into the first 1/3
    #if MC_bunchTimes[i] > centers[25]:
    if MC_bunchTimes[i] > centers[80]:
        continue
        
    # Only one prompt cluster
    skip_event = False
    for k in range(i+1,len(MC_cluster_time)):        # look at the subsequent clusters in this event
        if MC_cluster_number[k] != 0:   # if the next cluster is a secondary cluster
            # check if the time of the next cluster is within the prompt 2us window
            if MC_bunchTimes[k] < bunch_time_cutoff + time_to_prompt_end:
                skip_event = True
                break
        else:     # we found an event without a secondary cluster
            break
    if skip_event:
        continue  # Skip the entire event
        
    # 6. CCB cut
    if MC_cluster_Qb[i] >= 0.6:
        continue
        
    # 8. reconstructed FV cut
    if MC_recoVtx[i] < -50:
        continue
    r2 = MC_recoVtx[i]**2 + MC_recoVtz[i]**2
    if r2 > FV_radius**2 or np.abs(MC_recoVty[i]) > FV_half_height:
        continue
        
    # 7. Energy reco cut
    recoE = reco_energy(MC_cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(MC_hitX[i], MC_hitY[i], MC_hitZ[i], MC_hitPE[i], MC_hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
    
    if bunch_cutting(MC_bunchTimes[i], centers) == True:
        
        # event categorization
        if MC_External[i] == True:
            indy = 5
            MC_reco_E[indy].append(recoE)
            MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
            MC_bunch_times[indy].append(MC_bunchTimes[i])
            MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
            MC_hits[indy].append(MC_cluster_Hits[i])
            MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
        else:
            if MC_FV[i] == True:
                if MC_NC[i] == 1:       # NC
                    if MC_QEL[i] == 1:
                        if MC_Z[i] == 8:
                            if MC_PDG[i] == 14 or MC_PDG[i] == 12:      # nu (mu + e)
                                # nuNCQE
                                indy = 0
                                MC_reco_E[indy].append(recoE)
                                MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                                MC_bunch_times[indy].append(MC_bunchTimes[i])
                                MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                                MC_hits[indy].append(MC_cluster_Hits[i])
                                MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
                            elif MC_PDG[i] == -14 or MC_PDG[i] == -12:
                                # nubarNCQE
                                indy = 3
                                MC_reco_E[indy].append(recoE)
                                MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                                MC_bunch_times[indy].append(MC_bunchTimes[i])
                                MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                                MC_hits[indy].append(MC_cluster_Hits[i])
                                MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
                        else:
                            # NC elastic on H (NCother)
                            indy = 1
                            MC_reco_E[indy].append(recoE)
                            MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                            MC_bunch_times[indy].append(MC_bunchTimes[i])
                            MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                            MC_hits[indy].append(MC_cluster_Hits[i])
                            MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
                    else:
                        # NC, not QEL (NCother)
                        indy = 1
                        MC_reco_E[indy].append(recoE)
                        MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                        MC_bunch_times[indy].append(MC_bunchTimes[i])
                        MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                        MC_hits[indy].append(MC_cluster_Hits[i])
                        MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
                else:
                    # CC
                    indy = 2
                    MC_reco_E[indy].append(recoE)
                    MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                    MC_bunch_times[indy].append(MC_bunchTimes[i])
                    MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                    MC_hits[indy].append(MC_cluster_Hits[i])
                    MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
            else:
                # out-of-FV
                indy = 4
                MC_reco_E[indy].append(recoE)
                MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                MC_bunch_times[indy].append(MC_bunchTimes[i])
                MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                MC_hits[indy].append(MC_cluster_Hits[i])
                MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
                
               
    # outside bunches
    else:
        # event categorization
        if MC_External[i] == True:
            indy = 5
            MC_outside_weights_by_cat[indy].append(MC_CV_weight[i][0])
            MC_outside_bunch_by_cat[indy].append(MC_bunchTimes[i])
        else:
            if MC_FV[i] == True:
                if MC_NC[i] == 1:       # NC
                    if MC_QEL[i] == 1:
                        if MC_Z[i] == 8:
                            if MC_PDG[i] == 14 or MC_PDG[i] == 12:      # nu (mu + e)
                                # nuNCQE
                                indy = 0
                                MC_outside_weights_by_cat[indy].append(MC_CV_weight[i][0])
                                MC_outside_bunch_by_cat[indy].append(MC_bunchTimes[i])
                            elif MC_PDG[i] == -14 or MC_PDG[i] == -12:
                                # nubarNCQE
                                indy = 3
                                MC_outside_weights_by_cat[indy].append(MC_CV_weight[i][0])
                                MC_outside_bunch_by_cat[indy].append(MC_bunchTimes[i])
                        else:
                            # NC elastic on H (NCother)
                            indy = 1
                            MC_outside_weights_by_cat[indy].append(MC_CV_weight[i][0])
                            MC_outside_bunch_by_cat[indy].append(MC_bunchTimes[i])
                    else:
                        # NC, not QEL (NCother)
                        indy = 1
                        MC_outside_weights_by_cat[indy].append(MC_CV_weight[i][0])
                        MC_outside_bunch_by_cat[indy].append(MC_bunchTimes[i])
                else:
                    # CC
                    indy = 2
                    MC_outside_weights_by_cat[indy].append(MC_CV_weight[i][0])
                    MC_outside_bunch_by_cat[indy].append(MC_bunchTimes[i])
            else:
                # out-of-FV
                indy = 4
                MC_outside_weights_by_cat[indy].append(MC_CV_weight[i][0])
                MC_outside_bunch_by_cat[indy].append(MC_bunchTimes[i])
            

print('done')

applying NCQE selection cuts to MC...: 100%|█| 1064828/1064828 [01:00<00:00, 176

done


In [201]:
# on-beam data

reco_E = []
CV_x = []
CV_y = []
CV_z = []
bunch_times = []
vertex_x = []
vertex_y = []
vertex_z = []
hits = []
reco_runs = []

# outside bunches
outside_bunch_by_cat = []

for i in trange(len(cluster_time), desc = 'applying NCQE selection cuts to on-beam data...'):
    
    # ------ Has cluster ------- #
    # extracted ntuples have been filtered to only include clusters >= 10 hits
    # only_prompt_cluster covers CN == 0 and the cluster is in the prompt window
    if only_prompt_cluster[i] == False:
        continue
    # ------ Has cluster ------- #
    
    # No FMV or MRD activity
    if TankMRDCoinc[i] == 1 or NoVeto[i] == 0 or MRD_activity[i] != 0 or Veto_activity[i] != 0:
        continue
        
    # CCB cut
    if cluster_Qb[i] >= 0.6:
        continue
        
    # reconstructed FV cut
    if Vtx_reco[i] < -50:
        continue
    r2 = Vtx_reco[i]**2 + Vtz_reco[i]**2
    if r2 > FV_radius**2 or np.abs(Vty_reco[i]) > FV_half_height:
        continue
        
    # Energy reco cut
    recoE = reco_energy(cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(hitX[i], hitY[i], hitZ[i], hitPE[i], hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
    if runs[i] > 4000:   # FY23
        if cluster_time[i] < spill_start_23:
            continue
        if cluster_time[i] > spill_end_23:
            continue
            
        # add for earlier time
        if cluster_time[i] > centers_data_23[25]:
            continue
        
        if bunch_cutting(cluster_time[i], centers_data_23) == False:
            outside_bunch_by_cat.append(cluster_time[i])
            continue
    else:
        if cluster_time[i] < spill_start_22:
            continue
        if cluster_time[i] > spill_end_22:
            continue
            
        # add for earlier time
        if cluster_time[i] > centers_data_22[25]:
            continue
        
        if bunch_cutting(cluster_time[i], centers_data_22) == False:
            outside_bunch_by_cat.append(cluster_time[i])
            continue
        
    # event selection
    reco_E.append(recoE)
    CV_x.append(cw[0]); CV_y.append(cw[1]); CV_z.append(cw[2])
    bunch_times.append(cluster_time[i])
    vertex_x.append(Vtx_reco[i]); vertex_y.append(Vty_reco[i]); vertex_z.append(Vtz_reco[i])
    hits.append(cluster_Hits[i])
    reco_runs.append(runs[i])
          

print('done')

applying NCQE selection cuts to on-beam data...: 100%|█| 3542081/3542081 [00:33<

done


In [202]:
# ext data

ext_reco_E = []
ext_CV_x = []
ext_CV_y = []
ext_CV_z = []
ext_bunch_times = []
ext_vertex_x = []
ext_vertex_y = []
ext_vertex_z = []
ext_hits = []
ext_reco_runs = []

ext_outside_bunch_by_cat = []

for i in trange(len(cluster_time_ext), desc = 'applying NCQE selection cuts to off-beam data...'):
    
    # ------ Has cluster ------- #
    # extracted ntuples have been filtered to only include clusters >= 10 hits
    # only_prompt_cluster covers CN == 0 and the cluster is in the prompt window
    if only_prompt_cluster_ext[i] == False:
        continue
    # ------ Has cluster ------- #
    
    # No FMV or MRD activity
    if TankMRDCoinc_ext[i] == 1 or NoVeto_ext[i] == 0 or MRD_activity_ext[i] != 0 or Veto_activity_ext[i] != 0:
        continue
        
    # CCB cut
    if cluster_Qb_ext[i] >= 0.6:
        continue
        
    # reconstructed FV cut
    if Vtx_reco_ext[i] < -50:
        continue
    r2 = Vtx_reco_ext[i]**2 + Vtz_reco_ext[i]**2
    if r2 > FV_radius**2 or np.abs(Vty_reco_ext[i]) > FV_half_height:
        continue
        
    # Energy reco cut
    recoE = reco_energy(cluster_charge_ext[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(hitX_ext[i], hitY_ext[i], hitZ_ext[i], hitPE_ext[i], hitT_ext[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
    if runs_ext[i] > 4000:   # FY23
        if cluster_time_ext[i] < spill_start_23:
            continue
        if cluster_time_ext[i] > spill_end_23:
            continue
            
        # add for earlier time
        if cluster_time_ext[i] > centers_data_23[25]:
            continue
        
        if bunch_cutting(cluster_time_ext[i], centers_data_23) == False:
            ext_outside_bunch_by_cat.append(cluster_time[i])
            continue
    else:
        if cluster_time_ext[i] < spill_start_22:
            continue
        if cluster_time_ext[i] > spill_end_22:
            continue
            
        # add for earlier time
        if cluster_time_ext[i] > centers_data_22[25]:
            continue
        
        if bunch_cutting(cluster_time_ext[i], centers_data_22) == False:
            ext_outside_bunch_by_cat.append(cluster_time[i])
            continue
        
    # event selection
    ext_reco_E.append(recoE)
    ext_CV_x.append(cw[0]); ext_CV_y.append(cw[1]); ext_CV_z.append(cw[2])
    ext_bunch_times.append(cluster_time_ext[i])
    ext_vertex_x.append(Vtx_reco_ext[i]); ext_vertex_y.append(Vty_reco_ext[i]); ext_vertex_z.append(Vtz_reco_ext[i])
    ext_hits.append(cluster_Hits_ext[i])
    ext_reco_runs.append(runs_ext[i])
          

print('done')

applying NCQE selection cuts to off-beam data...: 100%|█| 286189/286189 [00:02<0

done


In [207]:
# --- 1. Scaling Constants ---
#POT_MC = POT/(1e20)    # enable if using the 1/3 spill cut for MC just like data

# for 1/3 spill scaling trickery:
POT_MC = (POT/(1e20))*(80.5/25.5)   # (or in other words, ~3.15)

POT_DATA = total_POT_TOR860
TRIG_DATA = total_triggers
TRIG_EXT = total_ext_triggers

EXPECTED_SKYSHINE = 15.7  # +8.8 / −12.4


# --- 2. Radius Calculation ---
# For MC (6 categories)
radius_mc = [np.sqrt(np.array(MC_vertex_x[i])**2 + np.array(MC_vertex_z[i])**2) for i in range(6)]
# For On-Beam Data
radius_data = np.sqrt(np.array(vertex_x)**2 + np.array(vertex_z)**2)
# For Off-Beam (EXT) Data
radius_ext = np.sqrt(np.array(ext_vertex_x)**2 + np.array(ext_vertex_z)**2)

# --- 3. Delta-T Calculation ---
def distance_to_bunch_center(cluster_time, centers):
    avg_diff = 0.0588
    # Ensure 'centers' is defined in your environment
    shifted_centers = centers - avg_diff 
    return np.min(np.abs(shifted_centers - cluster_time))

# --- 2. MC Calculation (Static) ---
# Assuming 'centers' is your MC bunch center array
delta_t_mc = [
    np.array([distance_to_bunch_center(t, centers) for t in cat]) 
    for cat in MC_bunch_times
]

# --- 3. On-Beam Data (Run Dependent) ---
delta_t_data_list = []
for t, run in zip(bunch_times, reco_runs):
    # Select epoch centers based on run number
    current_centers = centers_data_22 if run < 4000 else centers_data_23
    delta_t_data_list.append(distance_to_bunch_center(t, current_centers))

delta_t_data = np.array(delta_t_data_list)

# --- 4. Off-Beam Data (Run Dependent) ---
delta_t_ext_list = []
for t, run in zip(ext_bunch_times, ext_reco_runs):
    # Same logic for Off-beam/External data
    current_centers = centers_data_22 if run < 4000 else centers_data_23
    delta_t_ext_list.append(distance_to_bunch_center(t, current_centers))

delta_t_ext = np.array(delta_t_ext_list)

print('done')

done


In [208]:
def get_normalized_times(times, runs, weights=None, mc=False):

    norm_times = []
    norm_weights = []

    for i, (t, r) in enumerate(zip(times, runs)):

        if mc:
            t0 = centers[0]
        else:
            t0 = centers_data_22[0] if r < 4000 else centers_data_23[0]

        tnorm = t - t0

        # Filter to only include the 80 bunches
        if 0 <= tnorm < (80 * 18.936):

            norm_times.append(tnorm)

            if weights is not None:
                norm_weights.append(weights[i])

    if weights is not None:
        return np.array(norm_times), np.array(norm_weights)

    return np.array(norm_times)


# 1. Process MC (Categories 0-5)
bunches_80_mc = []
bunches_80_weights = []

for i in range(6):

    t, w = get_normalized_times(
        MC_bunch_times[i],
        [0]*len(MC_bunch_times[i]),
        weights=MC_weights_by_cat[i],
        mc=True
    )

    bunches_80_mc.append(t)
    bunches_80_weights.append(w)

# 2. Process On-Beam Data
bunches_80_data = get_normalized_times(bunch_times, reco_runs)

# 3. Process Off-Beam (External)
bunches_80_ext = get_normalized_times(ext_bunch_times, ext_reco_runs)

print('done')

done


In [197]:
# add skyshine

file_names = [
    '../skyshine_100k_no_ext_clusters.root',
    '../skyshine_100k_no_ext_clusters_2.root',
    '../skyshine_100k_no_ext_clusters_3.root'
]

# scalar accumulators
skyshine_cluster_number    = []
skyshine_cluster_time      = []
skyshine_cluster_charge    = []
skyshine_cluster_Qb        = []
skyshine_cluster_Hits      = []
skyshine_TankMRDCoinc      = []
skyshine_NoVeto            = []
skyshine_recoVtx           = []
skyshine_recoVty           = []
skyshine_recoVtz           = []
skyshine_event_number      = []

# jagged accumulators
skyshine_hitX  = []
skyshine_hitY  = []
skyshine_hitZ  = []
skyshine_hitT  = []
skyshine_hitPE = []
skyshine_hitID = []
skyshine_MRD_hitT  = []
skyshine_FMV_hitT  = []

for file_name in file_names:
    path = os.path.join(file_name)
    root = uproot.open(path)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    skyshine_cluster_number.append(T['clusterNumber'].array(library='np'))
    skyshine_cluster_charge.append(T['clusterPE'].array(library='np'))
    skyshine_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    skyshine_cluster_Hits.append(T['clusterHits'].array(library='np'))
    skyshine_cluster_time.append(T['clusterTime'].array(library='np'))
    skyshine_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    skyshine_NoVeto.append(T['NoVeto'].array(library='np'))
    skyshine_recoVtx.append(T['recoLeastSqVtxX'].array(library='np'))
    skyshine_recoVty.append(T['recoLeastSqVtxY'].array(library='np'))
    skyshine_recoVtz.append(T['recoLeastSqVtxZ'].array(library='np'))
    skyshine_event_number.append(CEN)

    skyshine_hitX.append(T['hitX'].array())
    skyshine_hitY.append(T['hitY'].array())
    skyshine_hitZ.append(T['hitZ'].array())
    skyshine_hitT.append(T['hitT'].array())
    skyshine_hitPE.append(T['hitPE'].array())
    skyshine_hitID.append(T['hitDetID'].array())

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']

    # map trigger-level quantities onto clusters via CEN
    mrd_t   = S['MRDhitT'].array()
    veto_t  = S['FMVhitT'].array()
    skyshine_MRD_hitT.append(mrd_t[CEN])
    skyshine_FMV_hitT.append(veto_t[CEN])

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

skyshine_cluster_number  = np.concatenate(skyshine_cluster_number)
skyshine_cluster_time    = np.concatenate(skyshine_cluster_time)
skyshine_cluster_charge  = np.concatenate(skyshine_cluster_charge)
skyshine_cluster_Qb      = np.concatenate(skyshine_cluster_Qb)
skyshine_cluster_Hits    = np.concatenate(skyshine_cluster_Hits)
skyshine_TankMRDCoinc    = np.concatenate(skyshine_TankMRDCoinc)
skyshine_NoVeto          = np.concatenate(skyshine_NoVeto)
skyshine_recoVtx         = np.concatenate(skyshine_recoVtx)
skyshine_recoVty         = np.concatenate(skyshine_recoVty)
skyshine_recoVtz         = np.concatenate(skyshine_recoVtz)
skyshine_event_number    = np.concatenate(skyshine_event_number)

skyshine_hitX     = ak.concatenate(skyshine_hitX)
skyshine_hitY     = ak.concatenate(skyshine_hitY)
skyshine_hitZ     = ak.concatenate(skyshine_hitZ)
skyshine_hitT     = ak.concatenate(skyshine_hitT)
skyshine_hitPE    = ak.concatenate(skyshine_hitPE)
skyshine_hitID    = ak.concatenate(skyshine_hitID)
skyshine_MRD_hitT = ak.concatenate(skyshine_MRD_hitT)
skyshine_FMV_hitT = ak.concatenate(skyshine_FMV_hitT)

print(f'Total clusters: {len(skyshine_cluster_number)}')
print('\ndone')


Managing arrays...

Total clusters: 23216

done


In [198]:
# make skyshine cuts

import random

skyshine_reco_E = []
skyshine_CV_x = []
skyshine_CV_y = []
skyshine_CV_z = []
skyshine_bunch_times = []
skyshine_vertex_x = []
skyshine_vertex_y = []
skyshine_vertex_z = []
skyshine_hits = []

skyshine_delta_t_bunch = []


# do not apply timing cuts
for i in trange(len(skyshine_cluster_time), desc = 'applying NCQE selection cuts to simulated skyshine...'):
    
    # ------ Has cluster ------- #
    # primary cluster
    if skyshine_cluster_number[i] != 0:
        continue
    # cluster is within the prompt window
    if skyshine_cluster_time[i] > bunch_time_cutoff + time_to_prompt_end:
        continue
    # minimum 10 hits
    if skyshine_cluster_Hits[i] < 10:
        continue
    # ------ Has cluster ------- #
        
    if (
        skyshine_TankMRDCoinc[i] == 1
        or skyshine_NoVeto[i] == 0
        or any(0 <= t <= 300 for t in skyshine_MRD_hitT[i])
        or any(0 <= t <= 300 for t in skyshine_FMV_hitT[i])
    ):
        continue
        
    # Cluster time within 1.6 us beam spill
    if skyshine_cluster_time[i] > bunch_time_cutoff:
        continue
        
    # Add earlier time cut
    #if MC_bunchTimes[i] > centers[25]:
    #    continue
        
    # Only one prompt cluster
    skip_event = False
    for k in range(i+1,len(skyshine_cluster_time)):        # look at the subsequent clusters in this event
        if skyshine_cluster_number[k] != 0:   # if the next cluster is a secondary cluster
            # check if the time of the next cluster is within the prompt 2us window
            if skyshine_cluster_time[k] < bunch_time_cutoff + time_to_prompt_end:
                skip_event = True
                break
        else:     # we found an event without a secondary cluster
            break
    if skip_event:
        continue  # Skip the entire event
        
    # 6. CCB cut
    if skyshine_cluster_Qb[i] >= 0.6:
        continue
        
    # 8. reconstructed FV cut
    if skyshine_recoVtx[i] < -50:
        continue
    r2 = skyshine_recoVtx[i]**2 + skyshine_recoVtz[i]**2
    if r2 > FV_radius**2 or np.abs(skyshine_recoVty[i]) > FV_half_height:
        continue
        
    # 7. Energy reco cut
    recoE = reco_energy(skyshine_cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    # INVERT BUNCH CUT FOR SIDEBAND
    #if bunch_cutting(MC_bunchTimes[i], centers) == True:
    #    continue
        
    cw = pairwise_relative_direction(skyshine_hitX[i], skyshine_hitY[i], skyshine_hitZ[i], skyshine_hitPE[i], skyshine_hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
        
    skyshine_reco_E.append(recoE)
    skyshine_CV_x.append(cw[0]); skyshine_CV_y.append(cw[1]); skyshine_CV_z.append(cw[2])
    skyshine_bunch_times.append(skyshine_cluster_time[i])
    skyshine_vertex_x.append(skyshine_recoVtx[i]); 
    skyshine_vertex_y.append(skyshine_recoVty[i]);
    skyshine_vertex_z.append(skyshine_recoVtz[i])
    skyshine_hits.append(skyshine_cluster_Hits[i])
    
    # impart random position WRT bunch (since its constant in time)
    #skyshine_delta_t_bunch.append(random.random()*3.59)
    
    # symmetric
    skyshine_delta_t_bunch.append(random.random()*3.59*2 - 3.59)
    
    
print(len(skyshine_bunch_times))
print('done')

applying NCQE selection cuts to simulated skyshine...: 100%|█| 23216/23216 [00:0

802
done


In [187]:
# plotting function for PMT observables with skyshine (no time dependence!)

def plot_with_skyshine(bins, mc_data, on_beam_data, off_beam_data, skyshine_data, 
                       pot_mc, pot_data, trig_ext, trig_data, expected_skyshine_yield,
                       axis_label, y_axis_label, x_limit, y_limit, xaxis_minor, 
                       plot_name, legend_columns=3, SAVE_FIG=True,mc_weights_by_cat=None):
    
    # 1. Scaling Factors
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext
    
    ### NEW: Calculate the weight required to force the skyshine MC to exactly match your expected yield
    if len(skyshine_data) > 0:
        f_sky = expected_skyshine_yield / len(skyshine_data)
    else:
        f_sky = 0.0
    
    # 2. Prepare Prediction Stack (MC + Skyshine + Off-Beam)
    ### NEW: We inject Skyshine right after External (MC) [which is index 0]
    prediction_data = [mc_data[0], skyshine_data] + mc_data[1:] + [off_beam_data]
    
   # --- MC weights including GENIE tuning weights ---
    mc_weights = []

    for i, arr in enumerate(mc_data):

        if mc_weights_by_cat is None:
            genie_w = np.ones(len(arr))
        else:
            genie_w = np.array(mc_weights_by_cat[i])

        mc_weights.append(genie_w * f_pot)

    # Assemble full weight list
    weights = [mc_weights[0],
               np.ones(len(skyshine_data)) * f_sky] + \
              mc_weights[1:] + \
              [np.ones(len(off_beam_data)) * f_ext]
    
    # 3. Aesthetics
    ### NEW: Added Skyshine to labels and colors and increased outlines to 7
    labels_base = ['External (MC)', 'Skyshine', 'Out-of-FV', 'CC', 'NCother', r'$\bar{\nu}$NCQE', r'$\nu$NCQE', 'Off-Beam Data']
    colors = ['papayawhip', 'darkmagenta', 'lime', 'darkred', 'cyan', 'darkorange', '#0066ff', 'gray']
    color_outline = ['navy'] * 7 + ['black']
    
    for i, label in enumerate(labels_base):
        if 'External (MC)' in label:
            weights[i] = weights[i] * 0.18
            print(f"Applied 0.18 scale to index {i} ({label})")
        ### NEW: Just a helpful printout to confirm the math worked
        if 'Skyshine' in label:
            print(f"Scaled {len(skyshine_data)} raw skyshine events to exactly {expected_skyshine_yield:.1f} events")

    # Calculate scaled yields for labels
    yields = [np.sum(w) for w in weights]
    total_pred_yield = sum(yields) if sum(yields) > 0 else 1
    labels = [f"{l} ({100*y/total_pred_yield:.1f}%)" for l, y in zip(labels_base, yields)]
    
    # 4. Math for Residuals and Errors
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_widths = np.diff(bins)
    
    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)
    
    pred_hist_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i])[0] for i in range(len(prediction_data))]
    pred_counts = np.sum(pred_hist_comps, axis=0)
    
    pred_w2_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)[0] for i in range(len(prediction_data))]
    pred_err = np.sqrt(np.sum(pred_w2_comps, axis=0))

    # --- Plotting ---
    fig = plt.figure(figsize=(7, 7))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.1)
    ax = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # A. The Fill
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            label=labels, color=colors, linewidth=0, alpha = 0.65)

    # B. The Outline
    #ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
    #        histtype='step', color=color_outline, linewidth=1)
    x_step = np.repeat(bins, 2)[1:-1]
    y_band_hi = np.repeat(pred_counts + pred_err, 2)
    y_band_lo = np.repeat(np.maximum(pred_counts - pred_err, 0), 2)
    ax.fill_between(x_step, y_band_lo, y_band_hi,
                    facecolor='none', edgecolor='dimgray',
                    linewidth=0, hatch='//', zorder=3,
                    label='MC stats')
    
    # C. Total prediction outline — CHANGED: single bold outline of the full stack
    #    (replaces the per-component stacked step outlines)
    #    ax.stairs requires matplotlib >= 3.4
    ax.stairs(pred_counts, bins, color='black', linewidth=1.8, zorder=4)
    
    
    # C. On-Beam Data Points
    ax.errorbar(bin_centers, data_counts, yerr=data_err, xerr=bin_widths/2, 
                fmt='ko', label='On-Beam Data (stats only)', markersize=4, capsize=0)

    # --- Residual Plot: (Data - Prediction) / Prediction ---
    mask = pred_counts > 0
    res = np.full_like(pred_counts, np.nan, dtype=float)
    res_err = np.full_like(pred_counts, np.nan, dtype=float)
    
    res[mask] = (data_counts[mask] - pred_counts[mask]) / pred_counts[mask]
    
    ratio = data_counts[mask] / pred_counts[mask]
    d_term = np.where(data_counts[mask] > 0, (data_err[mask]/data_counts[mask])**2, 0)
    res_err[mask] = ratio * np.sqrt(d_term + (pred_err[mask]/pred_counts[mask])**2)

    ax2.errorbar(bin_centers[mask], res[mask], yerr=res_err[mask], xerr=bin_widths[mask]/2, fmt='ko', markersize=4)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax2.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
    ax2.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)
    
    # Formatting
    ax.set_ylabel(y_axis_label)
    ax.set_xlim(x_limit)
    ax.set_ylim(y_limit)
    ax.tick_params(axis='x', labelbottom=False)
    ax.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax2.set_ylabel(r'$\frac{\mathrm{Data}}{\mathrm{Pred}} - 1$', fontsize=14)
    ax2.set_xlabel(axis_label)
    ax2.set_xlim(x_limit)
    ax2.set_ylim([-1.2, 1.2]) 
    ax2.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax.legend(ncol=legend_columns, fontsize=8, loc='upper right', frameon=False)
    
    ax.text(0.02, 1.02, 'ANNIE', fontweight='bold', color='blue', transform=ax.transAxes)
    ax.text(0.17, 1.02, 'Preliminary', transform=ax.transAxes)
    #ax.text(0.70, 1.02, f"{pot_data*100:.2f} " + r"$\times 10^{18}$ POT", transform=ax.transAxes)
    ax.text(0.70, 1.02, f"{pot_data:.2f} " + r"$\times 10^{20}$ POT", transform=ax.transAxes)

    plt.tight_layout()
    if SAVE_FIG:
        plt.savefig(plot_name, dpi=300, bbox_inches='tight')
    plt.show()


print('Plotting function loaded')

Plotting function loaded


In [174]:
path_folder = '../plots/FULL_DATA/spill_scaled/post_thesis/'
os.system('mkdir -p ' + path_folder)

SAVE_FIGURE = True

radius_skyshine = np.sqrt(np.array(skyshine_vertex_x)**2 + np.array(skyshine_vertex_z)**2)

# GENIE CV tuning weights
mc_weights_stack = [
    MC_weights_by_cat[5],
    MC_weights_by_cat[4],
    MC_weights_by_cat[2],
    MC_weights_by_cat[1],
    MC_weights_by_cat[3],
    MC_weights_by_cat[0]
]

common_args = {
    'pot_mc': POT_MC,
    'pot_data': POT_DATA,
    'trig_ext': TRIG_EXT,
    'trig_data': TRIG_DATA,
    'legend_columns': 3,
    'mc_weights_by_cat': mc_weights_stack
}

#'''
# energy
mc_stack = [MC_reco_E[5], MC_reco_E[4], MC_reco_E[2], MC_reco_E[1], MC_reco_E[3], MC_reco_E[0]]
plot_with_skyshine(
    bins=np.arange(5, 13, 1),
    mc_data=mc_stack,
    on_beam_data=reco_E,
    off_beam_data=ext_reco_E,
    skyshine_data=skyshine_reco_E, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, 
    axis_label='Reconstructed energy [MeV]',
    y_axis_label='Events / 1 MeV bin',
    x_limit=[5, 12],
    y_limit=[0,200],
    xaxis_minor=0.5,
    plot_name=path_folder + 'full_data_reco_energy.png',
    SAVE_FIG = SAVE_FIGURE,
    **common_args
)


mc_stack_hits = [MC_hits[5], MC_hits[4], MC_hits[2], MC_hits[1], MC_hits[3], MC_hits[0]]
plot_with_skyshine(
    bins=np.arange(10, 50, 4),
    mc_data=mc_stack_hits,
    on_beam_data=hits,
    off_beam_data=ext_hits,
    skyshine_data=skyshine_hits, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='PMT hits',
    y_axis_label='Events / 4 hit bin',
    x_limit=[10, 50],
    y_limit=[0,250],
    xaxis_minor=1,
    plot_name=path_folder + 'full_data_PMT_hits.png',
    SAVE_FIG = SAVE_FIGURE,
    **common_args
)

mc_stack_r = [radius_mc[5], radius_mc[4], radius_mc[2], radius_mc[1], radius_mc[3], radius_mc[0]]
plot_with_skyshine(
    bins=np.arange(0, 0.8, 0.1),
    mc_data=mc_stack_r,
    on_beam_data=radius_data,
    off_beam_data=radius_ext,
    skyshine_data=radius_skyshine, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Reconstructed distance to center [m]',
    y_axis_label='Events / 10 cm bin',
    x_limit=[0, 0.7],
    y_limit=[0,300],
    xaxis_minor=0.1,
    plot_name=path_folder + 'full_data_radius.png',
    SAVE_FIG = SAVE_FIGURE,
    **common_args
)

mc_stack_y = [MC_vertex_y[5], MC_vertex_y[4], MC_vertex_y[2], MC_vertex_y[1], MC_vertex_y[3], MC_vertex_y[0]]
plot_with_skyshine(
    bins=np.arange(-1, 1.25, 0.25),
    mc_data=mc_stack_y,
    on_beam_data=vertex_y,
    off_beam_data=ext_vertex_y,
    skyshine_data=skyshine_vertex_y, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Reconstructed vertex (Y) [m]',
    y_axis_label='Events / 50 cm bin',
    x_limit=[-1, 1],
    y_limit=[0,175],
    xaxis_minor=0.2,
    plot_name=path_folder + 'full_data_Y.png',
    SAVE_FIG = SAVE_FIGURE,
    **common_args
)

vars_cv = [('x', MC_CV_x, CV_x, ext_CV_x, skyshine_CV_x),
           ('y', MC_CV_y, CV_y, ext_CV_y, skyshine_CV_y),
           ('z', MC_CV_z, CV_z, ext_CV_z, skyshine_CV_z)]
for name, mc_list, data_list, ext_list, sky_list in vars_cv:
    mc_stack = [mc_list[5], mc_list[4], mc_list[2], mc_list[1], mc_list[3], mc_list[0]]
    plot_with_skyshine(
        bins=np.arange(-1, 1.25, 0.25) if name != 'z' else np.arange(-1, 0.6, 0.2),
        mc_data=mc_stack,
        on_beam_data=data_list,
        off_beam_data=ext_list,
        skyshine_data=sky_list, # <-- Pass the new skyshine simulated energies here
        expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
        axis_label=r'$C_V$ ' + f'({name})',
        y_axis_label='Events / bin',
        x_limit=[-1, 1] if name != 'z' else [-1, 0.4],
        y_limit=[0,200] if name != 'y' else [0, 225] ,
        xaxis_minor=0.1,
        plot_name=f'{path_folder}full_data_Cv_{name}.png',
        SAVE_FIG = SAVE_FIGURE,
        **common_args
    )
    
#'''
mc_stack = [delta_t_mc[5],
            delta_t_mc[4],
            delta_t_mc[2],
            delta_t_mc[1],
            delta_t_mc[3],
            delta_t_mc[0]]

plot_with_skyshine(
    bins=np.array([0,1,2,3,3.59]),
    mc_data=mc_stack,
    on_beam_data=delta_t_data,
    off_beam_data=delta_t_ext,
    skyshine_data=skyshine_delta_t_bunch, 
    expected_skyshine_yield=EXPECTED_SKYSHINE, 
    axis_label=r'$\Delta t_{\mathrm{nearest\ bunch}}$ [ns]',
    y_axis_label='Events / bin',
    x_limit=[0,3.59],
    y_limit=[0,375],
    xaxis_minor=0.25,
    plot_name=path_folder + 'full_data_delta_t.png',
    SAVE_FIG = SAVE_FIGURE,
    **common_args
)
#'''
    

print('done')

Applied 0.18 scale to index 0 (External (MC))
Scaled 802 raw skyshine events to exactly 15.7 events


/var/folders/nf/wq25_wtn08vb7t3wg0hydmhr0000gr/T/ipykernel_93085/1008424071.py:139: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Applied 0.18 scale to index 0 (External (MC))
Scaled 802 raw skyshine events to exactly 15.7 events
Applied 0.18 scale to index 0 (External (MC))
Scaled 802 raw skyshine events to exactly 15.7 events
Applied 0.18 scale to index 0 (External (MC))
Scaled 802 raw skyshine events to exactly 15.7 events
Applied 0.18 scale to index 0 (External (MC))
Scaled 802 raw skyshine events to exactly 15.7 events
Applied 0.18 scale to index 0 (External (MC))
Scaled 802 raw skyshine events to exactly 15.7 events
Applied 0.18 scale to index 0 (External (MC))
Scaled 802 raw skyshine events to exactly 15.7 events
Applied 0.18 scale to index 0 (External (MC))
Scaled 802 raw skyshine events to exactly 15.7 events
done


In [189]:
# the time since first bunch is obviously time dependent, so we need to be wise with how we sample skyshine

# 1. Data-driven parameters from your earlier fit
t0 = 331.1
t_max = 18.936 * 25  # Max time of your plot

# 2. Get the number of skyshine events in your MC sample
# (Replace 'skyshine_cluster_charge' with whatever your main skyshine array is called)
N_skyshine_mc = len(skyshine_cluster_charge)

# 3. Generate the synthetic times via Inverse Transform Sampling
u = np.random.uniform(0, 1, N_skyshine_mc)
skyshine_times = t0 + (t_max - t0) * np.sqrt(u)

# Note: If your plot needs to include the possibility of times extending 
# slightly past t_max due to bin width, just set t_max to the absolute upper edge of your bins.


# plotting
path_folder = '../plots/FULL_DATA/post_thesis/'
SAVE_FIGURE = True
bin_width = 18.936 * 5  # Buckets of 16 bunches as per your request
bins = np.arange(0, 18.936 * 25 + bin_width, bin_width)


mc_stack = [bunches_80_mc[5], bunches_80_mc[4], bunches_80_mc[2], 
            bunches_80_mc[1], bunches_80_mc[3], bunches_80_mc[0]]

mc_weights_stack = [
    bunches_80_weights[5],
    bunches_80_weights[4],
    bunches_80_weights[2],
    bunches_80_weights[1],
    bunches_80_weights[3],
    bunches_80_weights[0]
]

common_args = {
        'pot_mc': POT_MC,
        'pot_data': POT_DATA,
        'trig_ext': TRIG_EXT,
        'trig_data': TRIG_DATA,
        'legend_columns': 3,
        'mc_weights_by_cat': mc_weights_stack
    }

plot_with_skyshine(
    bins=bins,
    mc_data=mc_stack,
    on_beam_data=bunches_80_data,
    off_beam_data=bunches_80_ext,
    skyshine_data=skyshine_times, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Time since first bunch [ns]',
    y_axis_label='Events / 5 BNB bunches',
    x_limit=[0, 18.936 * 25],
    y_limit=[0,320],
    xaxis_minor=50,
    plot_name=path_folder + 'full_data_bunches_5.png',
    SAVE_FIG=SAVE_FIGURE,
    **common_args
)

print('done')

Applied 0.18 scale to index 0 (External (MC))
Scaled 23216 raw skyshine events to exactly 15.7 events


/var/folders/nf/wq25_wtn08vb7t3wg0hydmhr0000gr/T/ipykernel_93085/1008424071.py:139: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


done


In [177]:
# total number of reconstructed events:
print('Total reconstructed events: ' + str(len(reco_E)))

Total reconstructed events: 790


In [190]:
# Breakdown of the event category counts (need to apply weighting)
# (pass this along to the XS extraction script)

N_skyshine = 15.7    # full data
POT_MC = POT/(1e20)       # MC POT [e20]
#POT_MC = (80.5/25.5)*POT/(1e20)        # 1/3 spill trickery
POT_DATA = 2.573
TRIG_DATA = 60624559
TRIG_EXT = 24737351

pot_scale = POT_DATA / POT_MC
trig_scale = TRIG_DATA / TRIG_EXT

cat_labels = ['NCQE (signal)', 'NCother', 'CC', 'nubarNCQE', 'OOFV', 'External', 'Off-beam', 'Skyshine']

print('='*30)
print('MC RAW COUNTS')
print('='*30)
print('MC POT:', POT_MC)
print('Total NCQE true signal:', 10100)
print('-'*20)

# MC categories (with weights)
counter = 0
for i in range(len(MC_weights_by_cat)):
    if i == 5:
        print(cat_labels[i], round(0.18*sum(MC_weights_by_cat[i]), 2))
        counter += 0.18*sum(MC_weights_by_cat[i])
    else:
        print(cat_labels[i], round(sum(MC_weights_by_cat[i]), 2))
        counter += sum(MC_weights_by_cat[i])

print('Off-beam:', len(ext_reco_E)*trig_scale); counter += len(ext_reco_E)*trig_scale
print('Skyshine:', N_skyshine); counter += N_skyshine
print('-'*20)
print('Total Predicted:', counter)

# # # # # # # # # # # # # # # # # # # # # #

print('\n')
print('='*30)
print('POT SCALED')
print('='*30)
print('MC POT:', POT_MC)
print('Total NCQE true signal:', 10100*(POT_DATA/POT_MC))
print('-'*20)

# MC categories (with weights)
counter = 0
for i in range(len(MC_weights_by_cat)):
    if i == 5:
        print(cat_labels[i], round(0.18*(POT_DATA/POT_MC)*sum(MC_weights_by_cat[i]), 2))
        counter += 0.18*(POT_DATA/POT_MC)*sum(MC_weights_by_cat[i])
    else:
        print(cat_labels[i], round((POT_DATA/POT_MC)*sum(MC_weights_by_cat[i]), 2))
        counter += (POT_DATA/POT_MC)*sum(MC_weights_by_cat[i])

print('Off-beam:', len(ext_reco_E)*trig_scale); counter += len(ext_reco_E)*trig_scale
print('Skyshine:', N_skyshine); counter += N_skyshine
print('-'*20)
print('Total Predicted:', counter)

MC RAW COUNTS
MC POT: 2.3004542000000003
Total NCQE true signal: 10100
--------------------
NCQE (signal) 181.0
NCother 43.0
CC 30.45
nubarNCQE 2.0
OOFV 290.38
External 49.82
Off-beam: 115.18429248952323
Skyshine: 15.7
--------------------
Total Predicted: 727.5271932433786


POT SCALED
MC POT: 2.3004542000000003
Total NCQE true signal: 11296.59525497182
--------------------
NCQE (signal) 202.44
NCother 48.09
CC 34.05
nubarNCQE 2.24
OOFV 324.78
External 55.72
Off-beam: 115.18429248952323
Skyshine: 15.7
--------------------
Total Predicted: 798.214328288397


In [ ]:
## Additional PMT Observables if doing the 1/3 spill scaling

In [209]:
# bunch timing structure BUT have it scaled to the full MC spill

BUNCH_SPACING = 18.936  # ns
N_DATA_BUNCHES = 25     # int(476.78 / 18.936)
T_FOLD = N_DATA_BUNCHES * BUNCH_SPACING  # ~473.4 ns — exact integer multiple preserves bunch phase
FOLD_FACTOR = 3.15#(80 * BUNCH_SPACING) / T_FOLD  # ~3.19

path_folder = '../plots/FULL_DATA/spill_scaled/post_thesis/'

def get_normalized_times(times, runs, weights=None, mc=False, fold_to_data=False):
    norm_times = []
    norm_weights = []
    t_cut = T_FOLD if (fold_to_data and mc) else (80 * BUNCH_SPACING)

    for i, (t, r) in enumerate(zip(times, runs)):
        if mc:
            t0 = centers[0]
        else:
            t0 = centers_data_22[0] if r < 4000 else centers_data_23[0]
        tnorm = t - t0

        if fold_to_data and mc:
            tnorm = tnorm % T_FOLD  # fold full spill back into data window

        if 0 <= tnorm < t_cut:
            norm_times.append(tnorm)
            if weights is not None:
                w = weights[i]
                norm_weights.append(w)

    if weights is not None:
        return np.array(norm_times), np.array(norm_weights)
    return np.array(norm_times)


# 1. Process MC (Categories 0-5)
bunches_80_mc = []
bunches_80_weights = []

for i in range(6):

    t, w = get_normalized_times(
        MC_bunch_times[i],
        [0]*len(MC_bunch_times[i]),
        weights=MC_weights_by_cat[i],
        mc=True,
        fold_to_data=False#True
    )

    bunches_80_mc.append(t)
    bunches_80_weights.append(w)

# 2. Process On-Beam Data
bunches_80_data = get_normalized_times(bunch_times, reco_runs)

# 3. Process Off-Beam (External)
bunches_80_ext = get_normalized_times(ext_bunch_times, ext_reco_runs)


# # # # # # #
# scaled plotting

# the time since first bunch is obviously time dependent, so we need to be wise with how we sample skyshine

# 1. Data-driven parameters from your earlier fit
t0 = 331.1
t_max = 18.936 * 25  # Max time of your plot

# 2. Get the number of skyshine events in your MC sample
# (Replace 'skyshine_cluster_charge' with whatever your main skyshine array is called)
N_skyshine_mc = len(skyshine_cluster_charge)

# 3. Generate the synthetic times via Inverse Transform Sampling
u = np.random.uniform(0, 1, N_skyshine_mc)
skyshine_times = t0 + (t_max - t0) * np.sqrt(u)

# Note: If your plot needs to include the possibility of times extending 
# slightly past t_max due to bin width, just set t_max to the absolute upper edge of your bins.


# plotting
bin_width = 18.936 * 5  # Buckets of 16 bunches as per your request
bins = np.arange(0, 18.936 * 25 + bin_width, bin_width)


mc_stack = [bunches_80_mc[5], bunches_80_mc[4], bunches_80_mc[2], 
            bunches_80_mc[1], bunches_80_mc[3], bunches_80_mc[0]]

mc_weights_stack = [
    bunches_80_weights[5],
    bunches_80_weights[4],
    bunches_80_weights[2],
    bunches_80_weights[1],
    bunches_80_weights[3],
    bunches_80_weights[0]
]

POT_DATA_FULL = 2.573
EXPECTED_SKYSHINE_SIGNAL_FULL = 15.7

common_args = {
        'pot_mc': (POT_MC),
        'pot_data': POT_DATA,
        'trig_ext': TRIG_EXT,
        'trig_data': TRIG_DATA,
        'legend_columns': 3,
        'mc_weights_by_cat': mc_weights_stack
    }

plot_with_skyshine(
    bins=bins,
    mc_data=mc_stack,
    on_beam_data=bunches_80_data,
    off_beam_data=bunches_80_ext,
    skyshine_data=skyshine_times, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Time since first bunch [ns]',
    y_axis_label='Events / 5 BNB bunches',
    x_limit=[0, 18.936 * 25],
    y_limit=[0,320],
    xaxis_minor=50,
    plot_name=path_folder + 'full_data_bunches_5_folded.png',
    **common_args
)

print('done')

Applied 0.18 scale to index 0 (External (MC))
Scaled 23216 raw skyshine events to exactly 15.7 events


/var/folders/nf/wq25_wtn08vb7t3wg0hydmhr0000gr/T/ipykernel_93085/1008424071.py:139: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


done


In [212]:
# re-creating the miniboone-style bunch plot + backgrounds

def distance_to_bunch_center(cluster_time, centers):
    avg_diff = 0.0588
    # Ensure 'centers' is defined in your environment
    shifted_centers = centers - avg_diff 
    diffs = shifted_centers - cluster_time
    idx = np.argmin(np.abs(diffs))
    return diffs[idx]

delta_t_mc = [
    np.array([distance_to_bunch_center(t, centers) for t in cat]) 
    for cat in MC_bunch_times
]

delta_t_data_list = []
for t, run in zip(bunch_times, reco_runs):
    # Select epoch centers based on run number
    current_centers = centers_data_22 if run < 4000 else centers_data_23
    delta_t_data_list.append(distance_to_bunch_center(t, current_centers))

delta_t_data = np.array(delta_t_data_list)

delta_t_ext_list = []
for t, run in zip(ext_bunch_times, ext_reco_runs):
    # Same logic for Off-beam/External data
    current_centers = centers_data_22 if run < 4000 else centers_data_23
    delta_t_ext_list.append(distance_to_bunch_center(t, current_centers))

delta_t_ext = np.array(delta_t_ext_list)


# # # # # # # # # #

path_folder = '../plots/FULL_DATA/spill_scaled/post_thesis/'

# GENIE CV tuning weights
mc_weights_stack = [
    MC_weights_by_cat[5],
    MC_weights_by_cat[4],
    MC_weights_by_cat[2],
    MC_weights_by_cat[1],
    MC_weights_by_cat[3],
    MC_weights_by_cat[0]
]

mc_stack = [delta_t_mc[5],
            delta_t_mc[4],
            delta_t_mc[2],
            delta_t_mc[1],
            delta_t_mc[3],
            delta_t_mc[0]]


# NEW ARRAYS
mc_stack_outside         = [outside_bunch_by_cat[5],
                           outside_bunch_by_cat[4],
                           outside_bunch_by_cat[2],
                           outside_bunch_by_cat[1],
                           outside_bunch_by_cat[3],
                           outside_bunch_by_cat[0]]
mc_stack_outside_weights = [MC_outside_weights_by_cat[5],
                           MC_outside_weights_by_cat[4],
                           MC_outside_weights_by_cat[2],
                           MC_outside_weights_by_cat[1],
                           MC_outside_weights_by_cat[3],
                           MC_outside_weights_by_cat[0]]


EXPECTED_SKYSHINE_OUTSIDE = 15.7 * (299.778 / 183.09) #68.3 * (299.778 / 183.09)

def plot_with_skyshine_and_outside(
        bins, mc_data, on_beam_data, off_beam_data, skyshine_data,
        mc_data_outside, mc_weights_outside,
        on_beam_outside, off_beam_outside, skyshine_outside_data,
        pot_mc, pot_data, trig_ext, trig_data,
        expected_skyshine_yield, expected_skyshine_outside_yield,
        axis_label, y_axis_label, x_limit, y_limit, y_limit_outside, 
        xaxis_minor, yaxis_minor, yaxis_minor_outside,
        plot_name, legend_columns=3, mc_weights_by_cat=None, combined_errors=False):

    # ------------------------------------------------------------------ #
    # 1. Shared scaling factors
    # ------------------------------------------------------------------ #
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext
    f_sky      = expected_skyshine_yield / len(skyshine_data)          if len(skyshine_data)         > 0 else 0.0
    f_sky_out  = expected_skyshine_outside_yield / len(skyshine_outside_data) if len(skyshine_outside_data) > 0 else 0.0

    # ------------------------------------------------------------------ #
    # 2. INSIDE weights
    # ------------------------------------------------------------------ #
    mc_weights = []
    for i, arr in enumerate(mc_data):
        genie_w = np.array(mc_weights_by_cat[i]) if mc_weights_by_cat is not None else np.ones(len(arr))
        mc_weights.append(genie_w * f_pot)

    prediction_data = [mc_data[0], skyshine_data] + mc_data[1:] + [off_beam_data]
    weights = ([mc_weights[0], np.ones(len(skyshine_data)) * f_sky]
               + mc_weights[1:]
               + [np.ones(len(off_beam_data)) * f_ext])

    # ------------------------------------------------------------------ #
    # 3. OUTSIDE weights  (mc_weights_outside already carries GENIE weights)
    # ------------------------------------------------------------------ #
    outside_mc_w = [np.array(mc_weights_outside[i]) * f_pot for i in range(len(mc_data_outside))]
    outside_sky_w = np.ones(len(skyshine_outside_data)) * f_sky_out
    outside_ext_w = np.ones(len(off_beam_outside)) * f_ext

    # stacked order mirrors inside: [ExtMC, Skyshine, Out-of-FV, CC, NCother, nubar NCQE, nu NCQE, OffBeam]
    outside_pred_data    = [mc_data_outside[0], skyshine_outside_data] + mc_data_outside[1:] + [off_beam_outside]
    outside_pred_weights = ([outside_mc_w[0], outside_sky_w]
                            + outside_mc_w[1:]
                            + [outside_ext_w])

    # ------------------------------------------------------------------ #
    # 4. Aesthetics
    # ------------------------------------------------------------------ #
    labels_base  = ['External (MC)', 'Skyshine', 'Out-of-FV', 'CC',
                    'NCother', r'$\bar{\nu}$NCQE', r'$\nu$NCQE', 'Off-Beam Data']
    colors       = ['papayawhip', 'darkmagenta', 'lime', 'darkred',
                    'cyan', 'darkorange', '#0066ff', 'gray']
    color_outline = ['navy'] * 7 + ['black']

    # Apply 0.18 fudge to External (MC) — index 0 — in both regions
    weights[0] *= 0.18
    outside_pred_weights[0] *= 0.18

    # ------------------------------------------------------------------ #
    # 5. Inside histogram math
    # ------------------------------------------------------------------ #
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_widths  = np.diff(bins)

    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)

    pred_hist_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i])[0]
                       for i in range(len(prediction_data))]
    pred_counts = np.sum(pred_hist_comps, axis=0)

    pred_w2_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)[0]
                     for i in range(len(prediction_data))]
    pred_err = np.sqrt(np.sum(pred_w2_comps, axis=0))

    plot_err = np.sqrt(data_err**2 + pred_err**2) if combined_errors else data_err

    # ------------------------------------------------------------------ #
    # 6. Outside single-bin math
    # ------------------------------------------------------------------ #
    out_data_count  = len(on_beam_outside)
    out_data_err    = np.sqrt(out_data_count)
    out_yields      = [np.sum(w) for w in outside_pred_weights]   # per component
    out_pred_total  = sum(out_yields)
    out_pred_err    = np.sqrt(sum(np.sum(w**2) for w in outside_pred_weights))
    out_plot_err    = np.sqrt(out_data_err**2 + out_pred_err**2) if combined_errors else out_data_err

    # Legend labels with % (use inside yields for the shared legend)
    yields            = [np.sum(w) for w in weights]
    total_pred_yield  = sum(yields) if sum(yields) > 0 else 1
    labels            = [f"{l} ({100*y/total_pred_yield:.1f}%)"
                         for l, y in zip(labels_base, yields)]

    # ------------------------------------------------------------------ #
    # 7. Layout  — wide main | narrow outside, each with residual row
    # ------------------------------------------------------------------ #
    fig = plt.figure(figsize=(7, 7))
    gs  = fig.add_gridspec(2, 3,
                           height_ratios=[4, 1],
                           width_ratios=[10, 0.4, 2.5],
                           hspace=0.1, wspace=0.08)

    ax      = fig.add_subplot(gs[0, 0])   # main histogram
    ax2     = fig.add_subplot(gs[1, 0])   # main residual
    ax_out  = fig.add_subplot(gs[0, 2])   # outside bin
    ax_out2 = fig.add_subplot(gs[1, 2])   # outside residual

    # ------------------------------------------------------------------ #
    # 8. Main panel
    # ------------------------------------------------------------------ #
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights,
            label=labels, color=colors, linewidth=0)
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights,
            histtype='step', color=color_outline, linewidth=0.8)
    ax.errorbar(bin_centers, data_counts, yerr=plot_err, xerr=bin_widths/2,
                fmt='ko', label='On-Beam Data', markersize=4, capsize=0)

    # ------------------------------------------------------------------ #
    # 9. Main residual
    # ------------------------------------------------------------------ #
    mask    = pred_counts > 0
    res     = np.full_like(pred_counts, np.nan, dtype=float)
    res_err = np.full_like(pred_counts, np.nan, dtype=float)
    res[mask] = (data_counts[mask] - pred_counts[mask]) / pred_counts[mask]
    ratio     = data_counts[mask] / pred_counts[mask]
    d_term    = np.where(data_counts[mask] > 0, (data_err[mask]/data_counts[mask])**2, 0)
    res_err[mask] = ratio * np.sqrt(d_term + (pred_err[mask]/pred_counts[mask])**2)

    ax2.errorbar(bin_centers[mask], res[mask], yerr=res_err[mask],
                 xerr=bin_widths[mask]/2, fmt='ko', markersize=4)
    for ax_, ls in [(ax2, '--'), ]:
        ax2.axhline(0,    color='gray', linestyle='--', linewidth=1)
        ax2.axhline(0.5,  color='gray', linestyle=':',  linewidth=0.8)
        ax2.axhline(-0.5, color='gray', linestyle=':',  linewidth=0.8)

    # ------------------------------------------------------------------ #
    # 10. Outside panel — stacked bars at x=0.5 in a [0,1] dummy axis
    # ------------------------------------------------------------------ #
    bar_x, bar_w = 0.5, 0.7
    bottom = 0.0
    for yield_val, col, ecol in zip(out_yields, colors, color_outline):
        ax_out.bar(bar_x, yield_val, bar_w, bottom=bottom,
                   color=col, edgecolor=ecol, linewidth=0.8)
        bottom += yield_val
        
    out_total = sum(out_yields) if sum(out_yields) > 0 else 1
    bottom = 0.0
    for yield_val, col in zip(out_yields, colors):
        if yield_val / out_total > 0.05:  # skip tiny segments to avoid clutter
            ax_out.text(bar_x, bottom + yield_val / 2,
                        f'{100*yield_val/out_total:.1f}%',
                        ha='center', va='center', fontsize=8, #fontweight='bold',
                        color='black')
        bottom += yield_val
    
    ax_out.errorbar(bar_x, out_data_count, yerr=out_plot_err,
                    fmt='ko', markersize=4, capsize=0)

    # Outside residual
    if out_pred_total > 0:
        res_out     = (out_data_count - out_pred_total) / out_pred_total
        ratio_out   = out_data_count / out_pred_total
        d_term_out  = (out_data_err / out_data_count)**2 if out_data_count > 0 else 0
        res_err_out = ratio_out * np.sqrt(d_term_out + (out_pred_err / out_pred_total)**2)
        ax_out2.errorbar(bar_x, res_out, yerr=res_err_out, fmt='ko', markersize=4)

    ax_out2.axhline(0,    color='gray', linestyle='--', linewidth=1)
    ax_out2.axhline(0.5,  color='gray', linestyle=':',  linewidth=0.8)
    ax_out2.axhline(-0.5, color='gray', linestyle=':',  linewidth=0.8)

    # ------------------------------------------------------------------ #
    # 11. Formatting
    # ------------------------------------------------------------------ #
    ax.set_ylabel(y_axis_label, fontsize = 16)
    ax.set_xlim(x_limit)
    ax.set_ylim(y_limit)
    ax.tick_params(axis='x', labelbottom=False)
    ax.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    ax.yaxis.set_minor_locator(MultipleLocator(yaxis_minor))

    ax2.set_ylabel(r'$\frac{\mathrm{Data}}{\mathrm{Pred}} - 1$', fontsize=16)
    ax2.set_xlabel(axis_label, fontsize = 16)
    ax2.set_xlim(x_limit)
    ax2.set_ylim([-1.2, 1.2])
    ax2.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))

    ax_out.set_xlim([0, 1])
    ax_out.set_ylim(y_limit_outside)
    ax_out.tick_params(axis='x', labelbottom=False)
    ax_out.set_xticks([])
    ax_out.yaxis.set_minor_locator(MultipleLocator(yaxis_minor_outside))

    ax_out2.set_xlim([0, 1])
    ax_out2.set_ylim([-1.2, 1.2])
    ax_out2.set_xticks([0.5])
    ax_out2.set_xticklabels([r'$> \pm 1\sigma_{\nu\mathrm{NCQE}}$'], fontsize=16)

    ax.legend(ncol=legend_columns, fontsize=8, loc='upper right', frameon=False)

    ax.text(0.02, 1.02, 'ANNIE',       fontweight='bold', color='blue', transform=ax.transAxes)
    #ax.text(0.17, 1.02, 'Preliminary',                                   transform=ax.transAxes)
    ax.text(0.70, 1.02, f"{pot_data:.2f}" + r" $\times 10^{20}$ POT",   transform=ax.transAxes)

    #plt.savefig(plot_name, dpi=300, bbox_inches='tight')
    plt.show()
    
    
common_args = {
        'pot_mc': (POT_MC),
        'pot_data': POT_DATA,
        'trig_ext': TRIG_EXT,
        'trig_data': TRIG_DATA,
        'legend_columns': 3,
        'mc_weights_by_cat': mc_weights_stack
    }
    
    
plot_with_skyshine_and_outside(
    bins=np.array([-3.59, -2.5, -1.5, -0.5, 0.5, 1.5, 2.5, 3.59]),
    mc_data=mc_stack,
    on_beam_data=delta_t_data,
    off_beam_data=delta_t_ext,
    skyshine_data=skyshine_delta_t_bunch,
    mc_data_outside=mc_stack_outside,
    mc_weights_outside=mc_stack_outside_weights,
    on_beam_outside=outside_bunch_by_cat,           # your outside-bunch data events
    off_beam_outside=ext_outside_bunch_by_cat,           # your outside-bunch ext events
    skyshine_outside_data=skyshine_delta_t_bunch,
    expected_skyshine_yield=EXPECTED_SKYSHINE,
    expected_skyshine_outside_yield=EXPECTED_SKYSHINE * (299.778 / 183.09),   # expected skyshine outside
    axis_label=r'$\Delta t_{\mathrm{nearest\ bunch}}$ [ns]',
    y_axis_label='Events / bin',
    x_limit=[-3.59, 3.59],
    y_limit=[0, 200],
    y_limit_outside=[0,1000],
    xaxis_minor=0.5,
    yaxis_minor=5,
    yaxis_minor_outside=50,
    plot_name='../plots/FULL_DATA/post_thesis/full_data_delta_t_outside.png',
    **common_args
)

print('done')

done
